# 05 - Feature Extraction
Extract kinematic, orientation, and spatial features from pose keypoints.

In [ ]:
# ===== CONFIGURATION =====
# GitHub -- do not change
GITHUB_REPO_URL = "https://github.com/kaarthik-balakrishnan/LightningPoseTrack.git"
GIT_BRANCH = "main"

# Google Drive root -- change this to match your Drive structure
DRIVE_ROOT = "/content/drive/My Drive/PigBehavior"  # <-- SET THIS to your root

# Derived paths (change if your folders are at custom locations)
DRIVE_POSE_OUTPUTS = f"{DRIVE_ROOT}/pose_outputs"    # Input: pose prediction CSVs
DRIVE_FEATURES = f"{DRIVE_ROOT}/features"             # Output: feature CSVs

# Video/feeder parameters -- set these for your setup
FEEDER_X = None  # Feeder x coordinate (pixels)
FEEDER_Y = None  # Feeder y coordinate (pixels)
FRAME_WIDTH = 640   # Video frame width
FRAME_HEIGHT = 480  # Video frame height
FPS = 30.0  # Video frame rate

# Google Drive folder ID (for reference)
DRIVE_FOLDER_ID = "1X_41ZW3HfwVeft2lPld3XNqXsdxRDIwb"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, sys
REPO_DIR = "/content/LightningPoseTrack"
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

In [ ]:
!pip install --quiet pandas numpy scipy pyarrow scikit-learn imageio[ffmpeg]

In [ ]:
from pathlib import Path

pose_dir = Path(DRIVE_POSE_OUTPUTS)
pose_files = list(pose_dir.rglob("*.parquet"))
print(f"Found {len(pose_files)} pose files")
for f in pose_files[:10]:
    print(f"  {f.relative_to(pose_dir)}")
if len(pose_files) > 10:
    print(f"  ... and {len(pose_files) - 10} more")

In [ ]:
import pandas as pd
from tqdm.notebook import tqdm
from src.pose.clean_pose import clean_pose_df
from src.features.kinematics import extract_kinematics
from src.features.orientation import extract_orientation
from src.features.spatial import extract_spatial

features_dir = Path(DRIVE_FEATURES)
features_dir.mkdir(parents=True, exist_ok=True)

for pose_file in tqdm(pose_files, desc="Extracting features"):
    df = pd.read_parquet(pose_file)

    # Clean pose
    df_clean = clean_pose_df(df)

    # Kinematics
    kin = extract_kinematics(df_clean, FPS)

    # Orientation
    orient = extract_orientation(df_clean, FPS)

    # Spatial (feeder position must be set)
    feeder = (FEEDER_X, FEEDER_Y) if FEEDER_X is not None and FEEDER_Y is not None else None
    frame_size = (FRAME_WIDTH, FRAME_HEIGHT)
    spatial = extract_spatial(df_clean, feeder_position=feeder, frame_size=frame_size)

    # Combine all features
    combined = pd.concat([kin, orient, spatial], axis=1)

    # Save
    rel_path = pose_file.relative_to(pose_dir)
    out_path = features_dir / rel_path
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path = out_path.with_name(out_path.stem.replace("_pose", "_features") + ".parquet")
    combined.to_parquet(out_path)

print(f"\nFeatures saved to {features_dir}")

In [ ]:
# Display sample features
feature_files = list(features_dir.rglob("*.parquet"))
if feature_files:
    sample = pd.read_parquet(feature_files[0])
    print(f"Feature columns ({len(sample.columns)}):")
    print(list(sample.columns))
    print(f"\nShape: {sample.shape}")
    print(f"\nFirst 5 rows:")
    sample.head()